In [ ]:
!pip install -q sentence-transformers openai tqdm

In [ ]:
# Check semantic distance for 1k PKU-SafeRLHF samples

!pip install -q sentence-transformers

import os
import re
import pandas as pd
import numpy as np
from sentence_transformers import SentenceTransformer
from google.colab import drive

drive.mount("/content/drive")

# =========================
# Paths
# =========================

input_path = ""

output_path = ""

# =========================
# Load dataset
# =========================

df = pd.read_csv(input_path)

print("Dataset shape:", df.shape)
print("Columns:", df.columns.tolist())

required_cols = {"source_global_id", "source_local_id", "prompt", "chosen", "rejected"}

missing = required_cols - set(df.columns)
if missing:
    raise ValueError(f"Missing required columns: {missing}. Found columns: {df.columns.tolist()}")

# =========================
# Text cleaning
# =========================

_control_chars = re.compile(r"[\x00-\x08\x0B\x0C\x0E-\x1F\x7F]")

def sanitize_text(text):
    if text is None or pd.isna(text):
        return ""
    text = str(text)
    text = _control_chars.sub(" ", text)
    text = re.sub(r"\s+", " ", text).strip()
    return text

# For PKU-SafeRLHF, chosen/rejected are already response-only fields.
df["chosen_response"] = df["chosen"].apply(sanitize_text)
df["rejected_response"] = df["rejected"].apply(sanitize_text)
df["prompt"] = df["prompt"].apply(sanitize_text)

# =========================
# Load embedding model
# =========================

model_name = "sentence-transformers/all-mpnet-base-v2"
model = SentenceTransformer(model_name)

chosen_texts = df["chosen_response"].fillna("").tolist()
rejected_texts = df["rejected_response"].fillna("").tolist()

# =========================
# Compute embeddings
# =========================

chosen_embeddings = model.encode(
    chosen_texts,
    batch_size=32,
    show_progress_bar=True,
    convert_to_numpy=True,
    normalize_embeddings=True
)

rejected_embeddings = model.encode(
    rejected_texts,
    batch_size=32,
    show_progress_bar=True,
    convert_to_numpy=True,
    normalize_embeddings=True
)

# =========================
# Compute semantic similarity and distance
# =========================

semantic_similarity = np.sum(chosen_embeddings * rejected_embeddings, axis=1)
semantic_distance = 1.0 - semantic_similarity

df["semantic_similarity_chosen_rejected"] = semantic_similarity
df["semantic_distance_chosen_rejected"] = semantic_distance

# =========================
# Distance statistics
# =========================

lowest_distance = df["semantic_distance_chosen_rejected"].min()
highest_distance = df["semantic_distance_chosen_rejected"].max()
mean_distance = df["semantic_distance_chosen_rejected"].mean()
median_distance = df["semantic_distance_chosen_rejected"].median()
std_distance = df["semantic_distance_chosen_rejected"].std()

print("\nSemantic distance statistics:")
print(f"Lowest semantic distance : {lowest_distance:.6f}")
print(f"Highest semantic distance: {highest_distance:.6f}")
print(f"Mean semantic distance   : {mean_distance:.6f}")
print(f"Median semantic distance : {median_distance:.6f}")
print(f"Std semantic distance    : {std_distance:.6f}")

# =========================
# High / low labels using mean threshold
# =========================

df["semantic_distance_label"] = np.where(
    df["semantic_distance_chosen_rejected"] >= mean_distance,
    "semantic_distance_high",
    "semantic_distance_low"
)

print("\nSemantic distance label counts:")
print(df["semantic_distance_label"].value_counts())

# =========================
# Optional: rank by distance for inspection
# =========================

df["semantic_distance_rank_low_to_high"] = (
    df["semantic_distance_chosen_rejected"]
    .rank(method="first", ascending=True)
    .astype(int)
)

df["semantic_distance_rank_high_to_low"] = (
    df["semantic_distance_chosen_rejected"]
    .rank(method="first", ascending=False)
    .astype(int)
)

# =========================
# Save
# =========================

df.to_csv(output_path, index=False)

print("\nSaved semantic-distance labeled PKU-SafeRLHF dataset to:")
print(output_path)

# =========================
# Preview
# =========================

df[
    [
        "source_global_id",
        "source_local_id",
        "prompt",
        "chosen_response",
        "rejected_response",
        "semantic_similarity_chosen_rejected",
        "semantic_distance_chosen_rejected",
        "semantic_distance_label",
    ]
].head()

In [ ]:
# PKU-SafeRLHF: Rewrite rejected first for low-distance samples.
# If rejected rewriting does not increase semantic distance,
# then rewrite chosen response and check again.

!pip install -q sentence-transformers openai tqdm

import os
import re
import time
import numpy as np
import pandas as pd
from tqdm import tqdm
from getpass import getpass
from sentence_transformers import SentenceTransformer
from openai import OpenAI
from google.colab import drive

drive.mount("/content/drive")

# =========================
# Paths
# =========================

input_path = ""

output_path = ""

df = pd.read_csv(input_path)

print("Loaded dataset:", df.shape)
print("Columns:", df.columns.tolist())

# =========================
# OpenAI API setup
# =========================

os.environ["OPENAI_API_KEY"] = getpass("Enter your OpenAI API key: ")
client = OpenAI()

# =========================
# Embedding model
# =========================

embedder_name = "sentence-transformers/all-mpnet-base-v2"
embedder = SentenceTransformer(embedder_name)

# =========================
# Text helpers
# =========================

_control_chars = re.compile(r"[\x00-\x08\x0B\x0C\x0E-\x1F\x7F]")

def sanitize_text(text):
    if text is None or pd.isna(text):
        return ""
    text = str(text)
    text = _control_chars.sub(" ", text)
    text = re.sub(r"\s+", " ", text).strip()
    return text


def compute_semantic_distance(text_a, text_b):
    """
    Computes cosine semantic distance:
        distance = 1 - cosine_similarity
    """
    emb_a = embedder.encode(
        [sanitize_text(text_a)],
        convert_to_numpy=True,
        normalize_embeddings=True
    )

    emb_b = embedder.encode(
        [sanitize_text(text_b)],
        convert_to_numpy=True,
        normalize_embeddings=True
    )

    similarity = float(np.sum(emb_a[0] * emb_b[0]))
    distance = 1.0 - similarity

    return similarity, distance


# =========================
# Rewriting functions
# =========================

def rewrite_rejected_response(
    prompt,
    chosen_response,
    rejected_response,
    model_name="gpt-4.1",
    max_retries=5,
    sleep_time=2
):
    """
    Rewrite the rejected response while keeping it lower quality than the chosen response.
    """

    prompt_text = f"""
You are rewriting a rejected assistant response from a preference-learning dataset.

Goal:
Rewrite the rejected response so that it becomes more semantically different from the chosen response.

Important constraints:
1. Do NOT make the rejected response better than the chosen response.
2. Preserve the fact that this is still the rejected / lower-quality response.
3. Keep it plausible as an assistant response to the same user query.
4. Do NOT add harmful, unsafe, or policy-violating content.
5. Do NOT mention that this is a rewrite.
6. Return only the rewritten rejected response.

User prompt:
{prompt}

Chosen response:
{chosen_response}

Original rejected response:
{rejected_response}

Rewritten rejected response:
"""

    for attempt in range(max_retries):
        try:
            response = client.responses.create(
                model=model_name,
                input=prompt_text,
                temperature=0.9,
                max_output_tokens=512,
            )

            rewritten_text = response.output_text.strip()

            if len(rewritten_text) == 0:
                return rejected_response

            return rewritten_text

        except Exception as e:
            print(f"Rejected rewrite API attempt {attempt + 1} failed: {e}")
            time.sleep(sleep_time)

    return rejected_response


def rewrite_chosen_response(
    prompt,
    chosen_response,
    rejected_response,
    model_name="gpt-4.1",
    max_retries=5,
    sleep_time=2
):
    """
    Rewrite the chosen response while preserving that it remains the preferred response.
    """

    prompt_text = f"""
You are rewriting a chosen/preferred assistant response from a preference-learning dataset.

Goal:
Rewrite the chosen response so that it becomes more semantically different from the rejected response.

Important constraints:
1. The rewritten chosen response MUST remain better than the rejected response.
2. Preserve the helpful, safe, and preferred quality of the chosen response.
3. Keep it plausible as an assistant response to the same user query.
4. Do NOT make it unnecessarily verbose.
5. Do NOT add harmful, unsafe, or policy-violating content.
6. Do NOT mention that this is a rewrite.
7. Return only the rewritten chosen response.

User prompt:
{prompt}

Original chosen response:
{chosen_response}

Rejected response:
{rejected_response}

Rewritten chosen response:
"""

    for attempt in range(max_retries):
        try:
            response = client.responses.create(
                model=model_name,
                input=prompt_text,
                temperature=0.8,
                max_output_tokens=512,
            )

            rewritten_text = response.output_text.strip()

            if len(rewritten_text) == 0:
                return chosen_response

            return rewritten_text

        except Exception as e:
            print(f"Chosen rewrite API attempt {attempt + 1} failed: {e}")
            time.sleep(sleep_time)

    return chosen_response


# =========================
# Regeneration functions
# =========================

def regenerate_rejected_until_distance_increases(
    prompt,
    original_chosen_response,
    original_rejected_response,
    original_distance,
    model_name="gpt-4.1",
    max_generation_attempts=5
):
    """
    Stage 1:
    Try rewriting rejected response until:
        distance(original_chosen_response, candidate_rejected_response)
        > original_distance
    """

    best_rejected_response = original_rejected_response
    best_similarity, best_distance = compute_semantic_distance(
        original_chosen_response,
        original_rejected_response
    )
    best_gain = best_distance - original_distance

    for attempt in range(1, max_generation_attempts + 1):

        candidate_rejected_response = rewrite_rejected_response(
            prompt=prompt,
            chosen_response=original_chosen_response,
            rejected_response=original_rejected_response,
            model_name=model_name
        )

        candidate_similarity, candidate_distance = compute_semantic_distance(
            original_chosen_response,
            candidate_rejected_response
        )

        candidate_gain = candidate_distance - original_distance

        if candidate_distance > best_distance:
            best_rejected_response = candidate_rejected_response
            best_similarity = candidate_similarity
            best_distance = candidate_distance
            best_gain = candidate_gain

        if candidate_distance > original_distance:
            return {
                "kept": True,
                "rewrite_type": "rejected",
                "final_chosen_response": original_chosen_response,
                "final_rejected_response": candidate_rejected_response,
                "new_similarity": candidate_similarity,
                "new_distance": candidate_distance,
                "distance_gain": candidate_gain,
                "num_attempts": attempt,
                "best_failed_response": best_rejected_response,
                "best_failed_distance": best_distance,
                "best_failed_gain": best_gain,
            }

    return {
        "kept": False,
        "rewrite_type": "rejected_failed",
        "final_chosen_response": original_chosen_response,
        "final_rejected_response": original_rejected_response,
        "new_similarity": best_similarity,
        "new_distance": best_distance,
        "distance_gain": best_gain,
        "num_attempts": max_generation_attempts,
        "best_failed_response": best_rejected_response,
        "best_failed_distance": best_distance,
        "best_failed_gain": best_gain,
    }


def regenerate_chosen_until_distance_increases(
    prompt,
    original_chosen_response,
    original_rejected_response,
    original_distance,
    model_name="gpt-4.1",
    max_generation_attempts=5
):
    """
    Stage 2:
    If rejected rewriting fails, try rewriting chosen response until:
        distance(candidate_chosen_response, original_rejected_response)
        > original_distance
    """

    best_chosen_response = original_chosen_response
    best_similarity, best_distance = compute_semantic_distance(
        original_chosen_response,
        original_rejected_response
    )
    best_gain = best_distance - original_distance

    for attempt in range(1, max_generation_attempts + 1):

        candidate_chosen_response = rewrite_chosen_response(
            prompt=prompt,
            chosen_response=original_chosen_response,
            rejected_response=original_rejected_response,
            model_name=model_name
        )

        candidate_similarity, candidate_distance = compute_semantic_distance(
            candidate_chosen_response,
            original_rejected_response
        )

        candidate_gain = candidate_distance - original_distance

        if candidate_distance > best_distance:
            best_chosen_response = candidate_chosen_response
            best_similarity = candidate_similarity
            best_distance = candidate_distance
            best_gain = candidate_gain

        if candidate_distance > original_distance:
            return {
                "kept": True,
                "rewrite_type": "chosen",
                "final_chosen_response": candidate_chosen_response,
                "final_rejected_response": original_rejected_response,
                "new_similarity": candidate_similarity,
                "new_distance": candidate_distance,
                "distance_gain": candidate_gain,
                "num_attempts": attempt,
                "best_failed_response": best_chosen_response,
                "best_failed_distance": best_distance,
                "best_failed_gain": best_gain,
            }

    return {
        "kept": False,
        "rewrite_type": "chosen_failed",
        "final_chosen_response": original_chosen_response,
        "final_rejected_response": original_rejected_response,
        "new_similarity": best_similarity,
        "new_distance": best_distance,
        "distance_gain": best_gain,
        "num_attempts": max_generation_attempts,
        "best_failed_response": best_chosen_response,
        "best_failed_distance": best_distance,
        "best_failed_gain": best_gain,
    }


# =========================
# Prepare required columns
# =========================

required_columns = [
    "source_global_id",
    "source_local_id",
    "prompt",
    "chosen",
    "rejected",
    "chosen_response",
    "rejected_response",
    "semantic_distance_label",
    "semantic_distance_chosen_rejected"
]

for col in required_columns:
    if col not in df.columns:
        raise ValueError(f"Expected column '{col}' is missing. Found columns: {df.columns.tolist()}")

df["prompt"] = df["prompt"].apply(sanitize_text)

df["original_chosen_response"] = df["chosen_response"].apply(sanitize_text)
df["original_rejected_response"] = df["rejected_response"].apply(sanitize_text)

df["refined_chosen_response"] = df["original_chosen_response"]
df["refined_rejected_response"] = df["original_rejected_response"]

df["final_chosen_response"] = df["original_chosen_response"]
df["final_rejected_response"] = df["original_rejected_response"]

df["original_semantic_distance"] = df["semantic_distance_chosen_rejected"]
df["new_semantic_similarity"] = np.nan
df["new_semantic_distance"] = df["original_semantic_distance"]
df["semantic_distance_gain"] = 0.0

df["rewrite_attempted"] = False
df["rejected_rewrite_attempted"] = False
df["chosen_rewrite_attempted"] = False

df["refinement_kept"] = False
df["rewrite_type_kept"] = "none"

df["num_rejected_generation_attempts"] = 0
df["num_chosen_generation_attempts"] = 0
df["total_generation_attempts"] = 0

df["rejected_rewrite_failed"] = False
df["chosen_rewrite_failed"] = False

# =========================
# Choose low-distance rows
# =========================

low_distance_indices = df.index[
    df["semantic_distance_label"] == "semantic_distance_low"
].tolist()

print("\nNumber of semantic_distance_low rows:", len(low_distance_indices))

# For testing, set a small number like 20.
# For all low-distance rows, keep this as None.
MAX_ROWS_TO_REWRITE = None

if MAX_ROWS_TO_REWRITE is not None:
    low_distance_indices = low_distance_indices[:MAX_ROWS_TO_REWRITE]

print("Number of rows selected for rewriting:", len(low_distance_indices))

MAX_REJECTED_REWRITE_ATTEMPTS = 5
MAX_CHOSEN_REWRITE_ATTEMPTS = 5

# =========================
# Two-stage rewrite loop
# =========================

for idx in tqdm(low_distance_indices):

    prompt = df.loc[idx, "prompt"]
    original_chosen_response = df.loc[idx, "original_chosen_response"]
    original_rejected_response = df.loc[idx, "original_rejected_response"]
    original_distance = float(df.loc[idx, "original_semantic_distance"])

    df.loc[idx, "rewrite_attempted"] = True

    # -------------------------
    # Stage 1: rewrite rejected
    # -------------------------

    rejected_result = regenerate_rejected_until_distance_increases(
        prompt=prompt,
        original_chosen_response=original_chosen_response,
        original_rejected_response=original_rejected_response,
        original_distance=original_distance,
        model_name="gpt-4.1",
        max_generation_attempts=MAX_REJECTED_REWRITE_ATTEMPTS
    )

    df.loc[idx, "rejected_rewrite_attempted"] = True
    df.loc[idx, "num_rejected_generation_attempts"] = rejected_result["num_attempts"]

    if rejected_result["kept"]:

        df.loc[idx, "refinement_kept"] = True
        df.loc[idx, "rewrite_type_kept"] = "rejected"

        df.loc[idx, "refined_rejected_response"] = rejected_result["final_rejected_response"]
        df.loc[idx, "final_rejected_response"] = rejected_result["final_rejected_response"]

        df.loc[idx, "final_chosen_response"] = original_chosen_response
        df.loc[idx, "refined_chosen_response"] = original_chosen_response

        df.loc[idx, "new_semantic_similarity"] = rejected_result["new_similarity"]
        df.loc[idx, "new_semantic_distance"] = rejected_result["new_distance"]
        df.loc[idx, "semantic_distance_gain"] = rejected_result["distance_gain"]

        df.loc[idx, "total_generation_attempts"] = rejected_result["num_attempts"]

        continue

    df.loc[idx, "rejected_rewrite_failed"] = True
    df.loc[idx, "refined_rejected_response"] = rejected_result["best_failed_response"]

    # -------------------------
    # Stage 2: rewrite chosen
    # -------------------------

    chosen_result = regenerate_chosen_until_distance_increases(
        prompt=prompt,
        original_chosen_response=original_chosen_response,
        original_rejected_response=original_rejected_response,
        original_distance=original_distance,
        model_name="gpt-4.1",
        max_generation_attempts=MAX_CHOSEN_REWRITE_ATTEMPTS
    )

    df.loc[idx, "chosen_rewrite_attempted"] = True
    df.loc[idx, "num_chosen_generation_attempts"] = chosen_result["num_attempts"]

    if chosen_result["kept"]:

        df.loc[idx, "refinement_kept"] = True
        df.loc[idx, "rewrite_type_kept"] = "chosen"

        df.loc[idx, "refined_chosen_response"] = chosen_result["final_chosen_response"]
        df.loc[idx, "final_chosen_response"] = chosen_result["final_chosen_response"]

        df.loc[idx, "final_rejected_response"] = original_rejected_response

        df.loc[idx, "new_semantic_similarity"] = chosen_result["new_similarity"]
        df.loc[idx, "new_semantic_distance"] = chosen_result["new_distance"]
        df.loc[idx, "semantic_distance_gain"] = chosen_result["distance_gain"]

    else:
        df.loc[idx, "chosen_rewrite_failed"] = True
        df.loc[idx, "refinement_kept"] = False
        df.loc[idx, "rewrite_type_kept"] = "none"

        df.loc[idx, "refined_chosen_response"] = chosen_result["best_failed_response"]
        df.loc[idx, "final_chosen_response"] = original_chosen_response
        df.loc[idx, "final_rejected_response"] = original_rejected_response

        df.loc[idx, "new_semantic_similarity"] = chosen_result["new_similarity"]
        df.loc[idx, "new_semantic_distance"] = chosen_result["new_distance"]
        df.loc[idx, "semantic_distance_gain"] = chosen_result["distance_gain"]

    df.loc[idx, "total_generation_attempts"] = (
        df.loc[idx, "num_rejected_generation_attempts"]
        + df.loc[idx, "num_chosen_generation_attempts"]
    )

# =========================
# For rows not rewritten, fill values
# =========================

not_attempted_mask = df["rewrite_attempted"] == False

df.loc[not_attempted_mask, "new_semantic_distance"] = df.loc[
    not_attempted_mask, "original_semantic_distance"
]

df.loc[not_attempted_mask, "semantic_distance_gain"] = 0.0
df.loc[not_attempted_mask, "rewrite_type_kept"] = "none"

# =========================
# Final columns for PKU
# =========================

df["final_chosen"] = df["final_chosen_response"]
df["final_rejected"] = df["final_rejected_response"]

# =========================
# Sanity check final distance
# =========================

final_similarities = []
final_distances = []

for _, row in tqdm(df.iterrows(), total=len(df), desc="Computing final distances"):
    sim, dist = compute_semantic_distance(
        row["final_chosen_response"],
        row["final_rejected_response"]
    )
    final_similarities.append(sim)
    final_distances.append(dist)

df["final_semantic_similarity"] = final_similarities
df["final_semantic_distance"] = final_distances
df["final_semantic_distance_gain"] = (
    df["final_semantic_distance"] - df["original_semantic_distance"]
)

# =========================
# Safety filter:
# If a kept rewrite somehow does not improve final distance, revert it.
# =========================

bad_kept_mask = (
    (df["refinement_kept"] == True)
    & (df["final_semantic_distance"] <= df["original_semantic_distance"])
)

print("\nBad kept rows before safety correction:", int(bad_kept_mask.sum()))

df.loc[bad_kept_mask, "final_chosen_response"] = df.loc[bad_kept_mask, "original_chosen_response"]
df.loc[bad_kept_mask, "final_rejected_response"] = df.loc[bad_kept_mask, "original_rejected_response"]
df.loc[bad_kept_mask, "final_chosen"] = df.loc[bad_kept_mask, "original_chosen_response"]
df.loc[bad_kept_mask, "final_rejected"] = df.loc[bad_kept_mask, "original_rejected_response"]
df.loc[bad_kept_mask, "refinement_kept"] = False
df.loc[bad_kept_mask, "rewrite_type_kept"] = "none"

# Recompute final distances after safety correction
final_similarities = []
final_distances = []

for _, row in tqdm(df.iterrows(), total=len(df), desc="Recomputing final distances"):
    sim, dist = compute_semantic_distance(
        row["final_chosen_response"],
        row["final_rejected_response"]
    )
    final_similarities.append(sim)
    final_distances.append(dist)

df["final_semantic_similarity"] = final_similarities
df["final_semantic_distance"] = final_distances
df["final_semantic_distance_gain"] = (
    df["final_semantic_distance"] - df["original_semantic_distance"]
)

# =========================
# Report statistics
# =========================

print("\nOriginal semantic distance statistics:")
print(f"Lowest original distance : {df['original_semantic_distance'].min():.6f}")
print(f"Highest original distance: {df['original_semantic_distance'].max():.6f}")
print(f"Mean original distance   : {df['original_semantic_distance'].mean():.6f}")

print("\nFinal semantic distance statistics:")
print(f"Lowest final distance : {df['final_semantic_distance'].min():.6f}")
print(f"Highest final distance: {df['final_semantic_distance'].max():.6f}")
print(f"Mean final distance   : {df['final_semantic_distance'].mean():.6f}")

print("\nFinal semantic distance gain statistics:")
print(df["final_semantic_distance_gain"].describe())

print("\nRewrite summary:")
print("Total rows:", len(df))
print("Rewrite attempted:", int(df["rewrite_attempted"].sum()))
print("Refinement kept:", int(df["refinement_kept"].sum()))

print("\nKept rewrite type counts:")
print(df["rewrite_type_kept"].value_counts())

print("\nRejected rewrite attempted:", int(df["rejected_rewrite_attempted"].sum()))
print("Chosen rewrite attempted:", int(df["chosen_rewrite_attempted"].sum()))

print("\nRejected rewrite failed:", int(df["rejected_rewrite_failed"].sum()))
print("Chosen rewrite failed:", int(df["chosen_rewrite_failed"].sum()))

if df["rewrite_attempted"].sum() > 0:
    print("\nAverage total generation attempts among attempted rows:")
    print(df.loc[df["rewrite_attempted"], "total_generation_attempts"].mean())

if df["refinement_kept"].sum() > 0:
    print("\nAverage final gain among kept rewrites:")
    print(df.loc[df["refinement_kept"], "final_semantic_distance_gain"].mean())

print("\nRows where final distance improved:")
print(int((df["final_semantic_distance"] > df["original_semantic_distance"]).sum()))

print("\nRows where final distance did not improve:")
print(int((df["final_semantic_distance"] <= df["original_semantic_distance"]).sum()))

# =========================
# Save dataset
# =========================

df.to_csv(output_path, index=False)

print("\nSaved refined PKU-SafeRLHF dataset to:")
print(output_path)

# =========================
# Preview
# =========================

preview_cols = [
    "source_global_id",
    "source_local_id",
    "prompt",
    "original_chosen_response",
    "original_rejected_response",
    "refined_chosen_response",
    "refined_rejected_response",
    "final_chosen_response",
    "final_rejected_response",
    "semantic_distance_label",
    "original_semantic_distance",
    "final_semantic_distance",
    "final_semantic_distance_gain",
    "rewrite_attempted",
    "refinement_kept",
    "rewrite_type_kept",
    "num_rejected_generation_attempts",
    "num_chosen_generation_attempts",
    "total_generation_attempts"
]

df[preview_cols].head()

In [ ]:
# PKU-SafeRLHF: create semantic-distance-aware augmented synthetic dataset
# Input: rewritten semantic-distance PKU 1k file
# Output: pku_saferlhf_1k_semantic_refined_aug26.csv

!pip install -q transformers datasets sentencepiece accelerate pandas tqdm

import os
import re
import math
import pandas as pd
from tqdm import tqdm

import torch
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM
from google.colab import drive

drive.mount("/content/drive")

# =========================
# CONFIG
# =========================

PARAPHRASER_MODEL = "humarin/chatgpt_paraphraser_on_T5_base"

IN_CSV = ""

OUT_DIR = ""
OUT_CSV = os.path.join(
    OUT_DIR,
    ""
)

N_PARAS = 5
BATCH_SIZE = 16

MAX_INPUT_CHARS = 6000
MAX_INPUT_TOKENS = 512
MAX_NEW_TOKENS_PROMPT = 128
MAX_NEW_TOKENS_RESPONSE = 192

NUM_BEAMS = 5
DO_SAMPLE = False
NO_REPEAT_NGRAM_SIZE = 3

KEEP_LOW_FAILED_ORIGINAL = True

# =========================
# Helpers
# =========================

_control_chars = re.compile(r"[\x00-\x08\x0B\x0C\x0E-\x1F\x7F]")

def sanitize_text(s):
    if s is None:
        return ""
    s = _control_chars.sub(" ", str(s))
    s = re.sub(r"\s+", " ", s).strip()
    return s


def clip_chars(s, max_chars):
    s = sanitize_text(s)

    if len(s) <= max_chars:
        return s

    head = s[: max_chars // 2]
    tail = s[-(max_chars // 2):]

    return sanitize_text(head + " ... " + tail)


def safe_bool(x):
    if isinstance(x, bool):
        return x

    if pd.isna(x):
        return False

    return str(x).strip().lower() in ["true", "1", "yes"]


def get_base_responses(row):
    semantic_label = sanitize_text(row.get("semantic_distance_label", ""))
    refinement_kept = safe_bool(row.get("refinement_kept", False))

    original_chosen_response = sanitize_text(
        row.get("original_chosen_response", row.get("chosen_response", row.get("chosen", "")))
    )

    original_rejected_response = sanitize_text(
        row.get("original_rejected_response", row.get("rejected_response", row.get("rejected", "")))
    )

    final_chosen_response = sanitize_text(
        row.get("final_chosen_response", original_chosen_response)
    )

    final_rejected_response = sanitize_text(
        row.get("final_rejected_response", original_rejected_response)
    )

    if semantic_label == "semantic_distance_high":
        return original_chosen_response, original_rejected_response, "high_original"

    if semantic_label == "semantic_distance_low" and refinement_kept:
        return final_chosen_response, final_rejected_response, "low_refined_kept"

    if semantic_label == "semantic_distance_low" and not refinement_kept:
        return original_chosen_response, original_rejected_response, "low_failed_original"

    return original_chosen_response, original_rejected_response, "unknown_original"


@torch.inference_mode()
def paraphrase_batch(
    texts,
    tokenizer,
    model,
    device,
    n_paras=5,
    max_new_tokens=192,
):
    inputs = [f"paraphrase: {t}" for t in texts]

    enc = tokenizer(
        inputs,
        return_tensors="pt",
        padding=True,
        truncation=True,
        max_length=MAX_INPUT_TOKENS,
    ).to(device)

    gen = model.generate(
        **enc,
        num_beams=NUM_BEAMS,
        num_return_sequences=n_paras,
        do_sample=DO_SAMPLE,
        max_new_tokens=max_new_tokens,
        no_repeat_ngram_size=NO_REPEAT_NGRAM_SIZE,
        early_stopping=True,
    )

    decoded = tokenizer.batch_decode(gen, skip_special_tokens=True)
    decoded = [sanitize_text(x) for x in decoded]

    out = []

    for i in range(len(texts)):
        chunk = decoded[i * n_paras : (i + 1) * n_paras]

        seen = set()
        uniq = []

        for c in chunk:
            if c and c not in seen:
                uniq.append(c)
                seen.add(c)

        while len(uniq) < n_paras:
            uniq.append(uniq[-1] if uniq else "")

        out.append(uniq[:n_paras])

    return out

# =========================
# Load input
# =========================

if not os.path.exists(IN_CSV):
    raise FileNotFoundError(f"Input CSV not found: {IN_CSV}")

df = pd.read_csv(IN_CSV)

required_cols = {
    "source_global_id",
    "source_local_id",
    "prompt",
    "chosen",
    "rejected",
    "semantic_distance_label",
}

missing = required_cols - set(df.columns)

if missing:
    raise ValueError(
        f"Missing columns in input CSV: {missing}. Found columns: {list(df.columns)}"
    )

if "refinement_kept" not in df.columns:
    df["refinement_kept"] = False

os.makedirs(OUT_DIR, exist_ok=True)

print("Input rows:", len(df))
print("Output file:", OUT_CSV)

# =========================
# Select rows
# =========================

if KEEP_LOW_FAILED_ORIGINAL:
    df_source = df.copy()
else:
    df_source = df[
        (df["semantic_distance_label"] == "semantic_distance_high")
        |
        (
            (df["semantic_distance_label"] == "semantic_distance_low")
            & (df["refinement_kept"].apply(safe_bool) == True)
        )
    ].copy()

print("Rows used for augmentation:", len(df_source))

# =========================
# Build base columns
# =========================

base_prompts = []
base_chosen_responses = []
base_rejected_responses = []
semantic_base_sources = []

for _, row in df_source.iterrows():
    base_prompt = sanitize_text(row["prompt"])
    base_chosen_response, base_rejected_response, base_source = get_base_responses(row)

    base_prompts.append(base_prompt)
    base_chosen_responses.append(base_chosen_response)
    base_rejected_responses.append(base_rejected_response)
    semantic_base_sources.append(base_source)

df_source["base_prompt"] = base_prompts
df_source["base_chosen_response"] = base_chosen_responses
df_source["base_rejected_response"] = base_rejected_responses
df_source["semantic_base_source"] = semantic_base_sources

print("\nSemantic base source counts:")
print(df_source["semantic_base_source"].value_counts())

# =========================
# Load paraphraser
# =========================

device = "cuda" if torch.cuda.is_available() else "cpu"
print("\nUsing device:", device)

tokenizer = AutoTokenizer.from_pretrained(PARAPHRASER_MODEL)
model = AutoModelForSeq2SeqLM.from_pretrained(PARAPHRASER_MODEL).to(device)
model.eval()

# =========================
# Generate paraphrases
# =========================

rows_out = []

n = len(df_source)
num_batches = math.ceil(n / BATCH_SIZE)

for b in tqdm(range(num_batches), desc="Batches"):
    chunk = df_source.iloc[b * BATCH_SIZE : (b + 1) * BATCH_SIZE].copy()

    prompt_texts = [
        clip_chars(x, MAX_INPUT_CHARS)
        for x in chunk["base_prompt"].tolist()
    ]

    chosen_texts = [
        clip_chars(x, MAX_INPUT_CHARS)
        for x in chunk["base_chosen_response"].tolist()
    ]

    rejected_texts = [
        clip_chars(x, MAX_INPUT_CHARS)
        for x in chunk["base_rejected_response"].tolist()
    ]

    prompt_paras = paraphrase_batch(
        prompt_texts,
        tokenizer,
        model,
        device,
        n_paras=N_PARAS,
        max_new_tokens=MAX_NEW_TOKENS_PROMPT,
    )

    chosen_paras = paraphrase_batch(
        chosen_texts,
        tokenizer,
        model,
        device,
        n_paras=N_PARAS,
        max_new_tokens=MAX_NEW_TOKENS_RESPONSE,
    )

    rejected_paras = paraphrase_batch(
        rejected_texts,
        tokenizer,
        model,
        device,
        n_paras=N_PARAS,
        max_new_tokens=MAX_NEW_TOKENS_RESPONSE,
    )

    for i, row in chunk.reset_index(drop=True).iterrows():

        source_global_id = row["source_global_id"]
        source_local_id = row["source_local_id"]

        semantic_distance_label = sanitize_text(row.get("semantic_distance_label", ""))
        semantic_base_source = sanitize_text(row.get("semantic_base_source", ""))
        refinement_kept = safe_bool(row.get("refinement_kept", False))
        rewrite_type_kept = sanitize_text(row.get("rewrite_type_kept", "none"))

        original_semantic_distance = row.get(
            "original_semantic_distance",
            row.get("semantic_distance_chosen_rejected", None)
        )

        final_semantic_distance = row.get("final_semantic_distance", None)
        final_semantic_distance_gain = row.get("final_semantic_distance_gain", None)

        base_prompt = sanitize_text(row["base_prompt"])
        base_chosen_response = sanitize_text(row["base_chosen_response"])
        base_rejected_response = sanitize_text(row["base_rejected_response"])

        # -------------------------
        # Pair 0: base pair
        # -------------------------

        rows_out.append({
            "source_global_id": source_global_id,
            "source_local_id": source_local_id,
            "pair_type": "base",
            "pair_id": 0,

            "semantic_distance_label": semantic_distance_label,
            "semantic_base_source": semantic_base_source,
            "refinement_kept": refinement_kept,
            "rewrite_type_kept": rewrite_type_kept,

            "original_semantic_distance": original_semantic_distance,
            "final_semantic_distance": final_semantic_distance,
            "final_semantic_distance_gain": final_semantic_distance_gain,

            "prompt": base_prompt,
            "chosen": base_chosen_response,
            "rejected": base_rejected_response,

            "prompt_variant": "base",
            "chosen_variant": "base",
            "rejected_variant": "base",
        })

        # -------------------------
        # 25 paraphrase pairs
        # -------------------------

        pid = 1

        for ci in range(N_PARAS):
            for rj in range(N_PARAS):

                pi = (ci + rj) % N_PARAS

                prompt_para = sanitize_text(prompt_paras[i][pi])
                chosen_para = sanitize_text(chosen_paras[i][ci])
                rejected_para = sanitize_text(rejected_paras[i][rj])

                rows_out.append({
                    "source_global_id": source_global_id,
                    "source_local_id": source_local_id,
                    "pair_type": "para_prompt_chosen_rejected",
                    "pair_id": pid,

                    "semantic_distance_label": semantic_distance_label,
                    "semantic_base_source": semantic_base_source,
                    "refinement_kept": refinement_kept,
                    "rewrite_type_kept": rewrite_type_kept,

                    "original_semantic_distance": original_semantic_distance,
                    "final_semantic_distance": final_semantic_distance,
                    "final_semantic_distance_gain": final_semantic_distance_gain,

                    "prompt": prompt_para,
                    "chosen": chosen_para,
                    "rejected": rejected_para,

                    "prompt_variant": f"para_{pi + 1}",
                    "chosen_variant": f"para_{ci + 1}",
                    "rejected_variant": f"para_{rj + 1}",
                })

                pid += 1

# =========================
# Save
# =========================

df_out = pd.DataFrame(rows_out)
df_out.to_csv(OUT_CSV, index=False)

print(f"\nSaved augmented CSV with {len(df_out)} rows.")
print("Expected rows = 26 * N_sources =", 26 * len(df_source), "| Got:", len(df_out))

print("\nPair type counts:")
print(df_out["pair_type"].value_counts())

print("\nSemantic base source counts:")
print(df_out["semantic_base_source"].value_counts())

print("\nSaved to:")
print(OUT_CSV)

df_out.head()